### Setting up envs and imports


In [ ]:
import os
import sys

from dotenv import load_dotenv
from pageindex import PageIndexClient, utils

load_dotenv()
PAGE_INDEX_API_KEY = os.getenv("API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
pi_client = PageIndexClient(api_key="PAGE_INDEX_API_KEY")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
REGISTRY_PATH = "pdf_registry.json"
PDF_SOURCE_DIR = "Policy Documents Curated 15"
# print(pi_client)

if not PAGE_INDEX_API_KEY:
    print(
        "❌ Validation Failed: No 'API_KEY' entry detected within your local .env configuration."
    )
    sys.exit(1)

clean_key = PAGE_INDEX_API_KEY.strip().replace('"', "").replace("'", "")
pi_client = PageIndexClient(api_key=clean_key)


### Testing Groq API KEY


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    reasoning_effort="medium",
)

print(completion.choices[0].message.content)


### Testing NVIDIA NIM API Key


In [ ]:
import os
import sys

from openai import OpenAI

# Load from environment variable
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    print("❌ Error: NIM_API_KEY not found in environment variables")
    sys.exit(1)

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

completion = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    temperature=0.5,
    max_tokens=1024,
)

print(completion.choices[0].message.content)


### Submitting the pdfs to the pi_client


In [ ]:
import os

pdf_folder = PDF_SOURCE_DIR
if not os.path.isdir(pdf_folder):
    raise FileNotFoundError(f"Could not find folder: {PDF_SOURCE_DIR}")

pdf_files = [
    os.path.join(pdf_folder, f)
    for f in sorted(os.listdir(pdf_folder))
    if f.lower().endswith(".pdf")
]

doc_ids = {}  # stores filename -> doc_id

for pdf_path in pdf_files:
    response = pi_client.submit_document(pdf_path)
    doc_id = response["doc_id"]
    filename = os.path.basename(pdf_path)
    doc_ids[filename] = doc_id
    print(f"✅ Submitted: {filename} → {doc_id}")

print("\nAll doc_ids:", doc_ids)


#### Registry helper functions -> load and save => json


In [ ]:
import json


def load_registry() -> dict:
    if not os.path.exists(REGISTRY_PATH):
        return {}

    with open(REGISTRY_PATH, "r") as f:
        content = f.read().strip()

    if not content:  # file exists but is empty
        return {}

    return json.loads(content)


def save_to_registry(doc_id: str, pdf_path: str, description: str):
    registry = load_registry()
    registry[doc_id] = {
        "doc_id": doc_id,
        "filename": os.path.basename(pdf_path),
        "description": description,
    }
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)
    print(f"✅ Saved: {os.path.basename(pdf_path)} → {doc_id}")


### Multi-purpose function -> creating the json registry , routing the query to the registry


In [ ]:
def call_nim(prompt: str) -> str:
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY
    )
    completion = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return completion.choices[0].message.content  # pyright: ignore[reportReturnType]


### Building the json registry for pdfs


In [ ]:
import time


def generate_description(doc_id: str, filename: str) -> str:
    """
    Waits for PageIndex to finish processing, then feeds the node tree
    to NIM to auto-generate a registry description for that PDF.
    """
    print(f"⏳ Waiting for tree: {filename}...")
    while not pi_client.is_retrieval_ready(doc_id):
        time.sleep(3)

    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a tree structure of an insurance PDF document.
Each node has a title and summary describing what that section covers.

Write a single description (2-3 sentences) that answers:
"What specific questions can a user ask that this document would answer?"

Be specific — mention coverage types, procedures, limits, rules covered.
Do NOT write generic phrases like "this document covers various insurance topics."

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply with ONLY the description text, nothing else.
"""
    description = call_nim(prompt)
    print(f"✅ {filename}:\n   {description}\n")
    return description


##### Iterating over multiple doc_ids -> save into json


In [ ]:
for filename, doc_id in doc_ids.items():
    description = generate_description(doc_id, filename)
    save_to_registry(
        doc_id=doc_id,
        pdf_path=os.path.join(PDF_SOURCE_DIR, filename),
        description=description,
    )

print("\n🎉 Registry complete. Contents:")
print(json.dumps(load_registry(), indent=2))


### Query Router


In [ ]:
def route_query(query: str) -> dict:
    registry = load_registry()

    catalog = [
        {
            "doc_id": v["doc_id"],
            "filename": v["filename"],
            "description": v["description"],
        }
        for v in registry.values()
    ]

    prompt = f"""You are a document router for an insurance Q&A assistant.
A user has asked a question. Decide which documents to search and classify the question type.

User question: {query}

Available documents:
{json.dumps(catalog, indent=2)}

Classify the question into ONE of these types:
- "specific"   : the answer is inside one or more of the documents above
- "general"    : general insurance knowledge, no document needed
- "ambiguous"  : too vague to route

Reply ONLY with this JSON, nothing else:
{{
    "thinking": "<your reasoning>",
    "type": "specific",
    "doc_ids": ["doc_id_1"],
    "clarification": ""
}}

Rules:
- type "general"   → doc_ids must be []
- type "ambiguous" → doc_ids must be [], clarification must be a follow-up question
- type "specific"  → list ALL relevant doc_ids, clarification must be ""
"""

    result = call_nim(prompt)
    parsed = json.loads(result)

    print(f"🔍 Type     : {parsed['type']}")
    print(f"📄 Doc IDs  : {parsed['doc_ids']}")
    print(f"💭 Reasoning: {parsed['thinking']}")

    return parsed


##### Testing the routing


In [ ]:
# Test all three types
route_query("What happens if I miss a premium payment?")  # should be specific
route_query("What is term insurance?")  # should be general
route_query("Tell me about my policy")  # should be ambiguous

### Pipeline — Node Search per Routed Document


In [ ]:
def search_nodes(doc_id: str, query: str) -> list[str]:
    """
    Searches the node tree of a single PDF for content relevant to the query.
    Returns a list of text chunks tagged with source filename and page number.
    """
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"⚠️  Doc {doc_id} not ready — skipping.")
        return []

    registry = load_registry()
    filename = registry[doc_id]["filename"]

    # fetch tree for this specific doc
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a question and a tree structure of a document.
Each node contains a node id, title, and summary.
Find all nodes likely to contain the answer to the question.

Question: {query}

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply ONLY with this JSON:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

    result = json.loads(call_nim(prompt))
    node_map = utils.create_node_mapping(tree)

    chunks = []
    for node_id in result["node_list"]:
        if node_id not in node_map:
            continue
        node = node_map[node_id]
        chunks.append(
            f"[Source: {filename}, Page {node['page_index']}]\n{node['text']}"
        )

    print(f"  📑 {filename}: {len(chunks)} relevant node(s) found")
    return chunks


In [ ]:
# grab any doc_id from your registry to test
test_doc_id = next(iter(load_registry().keys()))
chunks = search_nodes(test_doc_id, "What is the death benefit?")
for chunk in chunks:
    print(chunk[:300])
    print("---")


In [ ]:
def ask(query: str):
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print("=" * 60)

    # Step 1: route — classify query and get relevant doc_ids
    routing = route_query(query)
    q_type = routing["type"]

    # Step 2: branch based on question type
    if q_type == "ambiguous":
        print(f"\nCould you clarify: {routing['clarification']}")
        return

    elif q_type == "general":
        print("\nGeneral question — answering from LLM knowledge.\n")
        answer = call_nim(f"""Answer this insurance question in simple plain language.
Start with a one-sentence summary. Avoid jargon.
Question: {query}""")
        utils.print_wrapped(answer)

    elif q_type == "specific":
        print(f"\nSearching {len(routing['doc_ids'])} document(s)...")

        # Step 3: search nodes in each routed doc and collect chunks
        all_chunks = []
        for doc_id in routing["doc_ids"]:
            chunks = search_nodes(doc_id, query)
            all_chunks.extend(chunks)

        if not all_chunks:
            print("No relevant content found.")
            return

        context = "\n\n---\n\n".join(all_chunks)

        # Step 4: generate answer from retrieved context
        answer = call_nim(f"""Answer the question based only on the context below.
If context comes from multiple documents, mention which document each point is from.

Question: {query}

Context:
{context}

Instructions:
- Use plain simple language, avoid legal jargon
- Use "you" and "your" instead of "the policyholder"
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to actually do or know
""")
        print("\n📝 Answer:\n")
        utils.print_wrapped(answer)


##### ask() testing


In [ ]:
ask("What happens if I miss a premium payment?")

### User input the query


In [ ]:
user_query = input("What is your Query? \n")
print(user_query)
ask(user_query)